# casmi26 FPNet full-train rebuild (Issue #82)
Fold-safe kit `train_fpnet.py` unchanged, `--holdout '' --hours 1000 --max_steps N`; other arguments at the kit defaults. Inputs are checked against published sha256 before training. Not for v3 evaluation (trains on v3 answers).


In [ ]:
SMOKE_STEPS = 2000   # 0 = production run
PROD_STEPS = 82855
GUARD_SECONDS = 1680
FIXED_ARGS = ['--holdout', '', '--hours', '1000']
EXPECT = {('dmitriigluzdov/casmi26-fold-safe-fpnet', 'train_fpnet.py'): '46b4dceebb2badeac78e0d5e55849169f5bef8da70a3adf1a7431fe34e603312', ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'split.parquet'): 'dd62549f81b9540890e0da17a06d0dcd216a2781b2c31687bd333445fe57d45e', ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'fp_bits.npy'): 'c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/fpnet.py'): '0ea6e372d1f045a877d350234c2ccf5d93fdd98cb23bd4c22376bf7e99547608', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/traindata.py'): 'cecb329d96362f11d16b40a26f1888f092610ff6202812408c13e2c293c6a2bc', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/spectra.py'): 'e2481761cdfb2515e8a945751f85f6b9e885680a8e974d7ac35c78bbf279ae74', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/chem.py'): '99474f23d8a67040b0c0f8c70e73592a6f7f656f50103e350ca473451e2169f1', ('ahmedberatozer/casmi26-v2-pool', 'fp_bits.npy'): 'c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677', ('ahmedberatozer/casmi26-v2-pool', 'train_structs.parquet'): 'f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_mz.npy'): '482332fec3fabcc46e06ce1b23d79e633af7009b8190a3dcd0a675121b42342b', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_it.npy'): '9e127f00eb9d4217403991304e69166adc9a36a5e624ce7fd1ddbf6b1417cc23', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_off.npy'): 'af2be108a532a278ec4563c023808b7fa19e40fdf4ea7dc888e980bb6f618e6c', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_meta.parquet'): '4f968592dc75b9bf2b6256dda394be42bad3cd07fd243ff0078101e066a44376'}
RECORD_ONLY = [('ahmedberatozer/casmi26-v2-pool', 'pool_meta.parquet'), ('ahmedberatozer/casmi26-v2-pool', 'pool_fp.npy')]
WORK_LINKS = {'spec_off.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_off.npy'), 'spec_mz.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_mz.npy'), 'spec_it.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_it.npy'), 'spec_meta.parquet': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_meta.parquet'), 'split.parquet': ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'split.parquet'), 'train_structs.parquet': ('ahmedberatozer/casmi26-v2-pool', 'train_structs.parquet'), 'pool_meta.parquet': ('ahmedberatozer/casmi26-v2-pool', 'pool_meta.parquet'), 'pool_fp.npy': ('ahmedberatozer/casmi26-v2-pool', 'pool_fp.npy'), 'fp_bits.npy': ('ahmedberatozer/casmi26-v2-pool', 'fp_bits.npy')}
REPORT_SRC = '"""Issue #82: キットの train_fpnet.py のログから it/s と時間を読み、本番 82,855 ステップの時間を外挿する。\n\nノートブック（kernels/fpnet_full_train）にこのファイルの中身を埋め込んで使う。外挿の式は Issue #82 の事前登録のとおり:\n    本番の時間 = 準備とデータの読み込みの時間 + 82,855 / 定常の it/s + 16 x 最後の評価にかかった時間\n定常の it/s = ログの step 1000 の行から最後の step の行までのステップの差 / 時刻の差。\n"""\nimport json\nimport re\n\nPROD_STEPS = 82855\nN_EVALS = 16\nSESSION_LIMIT_H = 11.0\nSTEADY_FROM = 1000\n\n_TS = r\'(\\d\\d):(\\d\\d):(\\d\\d)\'\n_STEP = re.compile(_TS + r\' step (\\d+) lr \\S+ .* ([0-9.]+) it/s queue (\\d+)\\s*$\')\n_DATA = re.compile(r\'\\[Data\\] train spectra ([\\d,]+) groups ([\\d,]+) pool ([\\d,]+) nbits (\\d+)\')\n_SETS = re.compile(_TS + r" (\\{.*\\}) train spectra (\\d+)\\s*$")\n_PARAMS = re.compile(_TS + r\' parameters (\\d+) \')\n_EVAL = re.compile(_TS + r\' EVAL (\\{.*\\})\\s*$\')\n\n\ndef _sec(h, m, s):\n    return int(h) * 3600 + int(m) * 60 + int(s)\n\n\ndef _unwrap(times):\n    """HH:MM:SS の秒を、日付をまたいでも増えるように直す。"""\n    out, add, prev = [], 0, None\n    for t in times:\n        if prev is not None and t + add < prev:\n            add += 86400\n        out.append(t + add)\n        prev = t + add\n    return out\n\n\ndef parse_log(text):\n    steps, params_t, eval_t, eval_rec, data, sets = [], None, None, None, None, None\n    for line in text.splitlines():\n        m = _STEP.match(line)\n        if m:\n            steps.append((_sec(*m.group(1, 2, 3)), int(m.group(4)), float(m.group(5)), int(m.group(6))))\n            continue\n        m = _DATA.search(line)\n        if m:\n            data = dict(train_spectra=int(m.group(1).replace(\',\', \'\')), groups=int(m.group(2).replace(\',\', \'\')),\n                        pool=int(m.group(3).replace(\',\', \'\')), nbits=int(m.group(4)))\n            continue\n        m = _SETS.match(line)\n        if m:\n            sets = dict(eval_sets=m.group(4), train_spectra=int(m.group(5)), t=_sec(*m.group(1, 2, 3)))\n            continue\n        m = _PARAMS.match(line)\n        if m:\n            params_t = _sec(*m.group(1, 2, 3))\n            continue\n        m = _EVAL.match(line)\n        if m:\n            eval_t = _sec(*m.group(1, 2, 3))\n            eval_rec = json.loads(m.group(4))\n    return dict(steps=steps, params_t=params_t, eval_t=eval_t, eval=eval_rec, data=data, sets=sets)\n\n\ndef steady_rate(steps, start=STEADY_FROM):\n    """step >= start の最初の行から最後の行までの (ステップの差 / 時刻の差)。行が足りなければ None。"""\n    if not steps:\n        return None\n    ts = _unwrap([s[0] for s in steps])\n    idx = [i for i, s in enumerate(steps) if s[1] >= start]\n    if len(idx) < 2:\n        return None\n    a, b = idx[0], len(steps) - 1\n    dt = ts[b] - ts[a]\n    return (steps[b][1] - steps[a][1]) / dt if dt > 0 else None\n\n\ndef extrapolate(setup_seconds, rate, eval_seconds, prod_steps=PROD_STEPS, n_evals=N_EVALS):\n    """事前登録の式。rate が無ければ None。"""\n    if not rate:\n        return None\n    total = setup_seconds + prod_steps / rate + n_evals * eval_seconds\n    return dict(setup_seconds=setup_seconds, steady_it_per_s=rate, eval_seconds=eval_seconds, prod_steps=prod_steps,\n                n_evals=n_evals, train_seconds=prod_steps / rate, total_seconds=total, total_hours=total / 3600.0,\n                one_session=bool(total / 3600.0 <= SESSION_LIMIT_H), session_limit_hours=SESSION_LIMIT_H)\n\n\ndef summarize(text, proc_start_sec_of_day):\n    """ログ全体から報告用の数をまとめる。proc_start_sec_of_day はログと同じ時計の、学習プロセスを起動した時刻（その日の秒）。"""\n    p = parse_log(text)\n    steps = p[\'steps\']\n    rate = steady_rate(steps)\n    setup = None\n    if p[\'params_t\'] is not None:\n        setup = p[\'params_t\'] - proc_start_sec_of_day\n        if setup < 0:\n            setup += 86400\n    eval_s, lower_bound = 0.0, True\n    if p[\'eval_t\'] is not None and steps:\n        eval_s = p[\'eval_t\'] - steps[-1][0]\n        if eval_s < 0:\n            eval_s += 86400\n        lower_bound = False\n    ext = extrapolate(setup or 0.0, rate, eval_s)\n    if ext is not None:\n        ext[\'lower_bound\'] = lower_bound or setup is None\n    return dict(last_step=steps[-1][1] if steps else 0, last_logged_it_per_s=steps[-1][2] if steps else None,\n                last_queue=steps[-1][3] if steps else None, min_queue=min(s[3] for s in steps) if steps else None,\n                data=p[\'data\'], sets=p[\'sets\'], final_eval=p[\'eval\'], extrapolation=ext)\n'


In [ ]:
# ---- Issue #82: Engine 1 FP bank rebuilt on all train with the fold-safe kit's train_fpnet.py (plan B) ----
import json, os, sys, time, hashlib, shutil, subprocess, threading, signal, platform
from pathlib import Path
T0 = time.time()
log = lambda *a: print(time.strftime('%H:%M:%S'), *a, flush=True)
SMOKE = SMOKE_STEPS > 0
STEPS = SMOKE_STEPS if SMOKE else PROD_STEPS
OUT = Path('/kaggle/working'); RUN = OUT / 'run'; WORK = RUN / 'work'; CODE = OUT / 'code'
WORK.mkdir(parents=True, exist_ok=True)
REPORT = dict(issue=82, recipe='plan B: fold-safe kit train_fpnet.py unchanged', smoke_steps=SMOKE_STEPS, max_steps=STEPS,
              fixed_args=FIXED_ARGS, prod_steps=PROD_STEPS, status='started')

def ds_root(slug):
    owner, name = slug.split('/')
    for p in (Path('/kaggle/input') / name, Path('/kaggle/input/datasets') / owner / name):
        if p.is_dir():
            return p
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_dir()]
    assert len(hits) == 1, (slug, hits)
    return hits[0]

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

def path_of(slug, rel):
    p = ds_root(slug) / rel
    assert p.is_file(), p
    return p

# 1) inputs: published sha256 must match before training
t = time.time(); hashes = {}
for (slug, rel), want in EXPECT.items():
    got = sha256(path_of(slug, rel)); hashes[f'{slug}/{rel}'] = got
    assert got == want, (slug, rel, got, want)
for slug, rel in RECORD_ONLY:
    hashes[f'{slug}/{rel}'] = sha256(path_of(slug, rel))
REPORT['input_sha256'] = hashes; REPORT['seconds_hash_inputs'] = round(time.time() - t, 1)
log('INPUTS OK', len(EXPECT), 'checked,', len(RECORD_ONLY), 'recorded', REPORT['seconds_hash_inputs'], 's')

# 2) run dir: work/ links to the published files; code = copy of v4b code/casmi (writable for numba cache)
for name, (slug, rel) in WORK_LINKS.items():
    dst = WORK / name
    if not dst.exists():
        os.symlink(path_of(slug, rel), dst)
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(ds_root('ahmedberatozer/casmi26-v4b-models') / 'code' / 'casmi', CODE / 'casmi')
for rel in ('fpnet.py', 'traindata.py', 'spectra.py', 'chem.py'):
    assert sha256(CODE / 'casmi' / rel) == EXPECT[('ahmedberatozer/casmi26-v4b-models', 'code/casmi/' + rel)], rel
TRAIN = path_of('dmitriigluzdov/casmi26-fold-safe-fpnet', 'train_fpnet.py')

# 3) environment
import torch
REPORT['env'] = dict(python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda,
                     gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                     gpu_capability=list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
                     gpu_count=torch.cuda.device_count(), vcpu_os_cpu_count=os.cpu_count(),
                     vcpu_affinity=len(os.sched_getaffinity(0)), workers=16, workers_note='kit default, not passed',
                     platform=platform.platform(), torch_config=torch.__config__.show())
try:
    REPORT['env']['mem_total_gb'] = round(int(open('/proc/meminfo').read().split('MemTotal:')[1].split()[0]) / 2**20, 2)
except Exception as e:
    REPORT['env']['mem_total_gb'] = repr(e)
log('ENV', {k: v for k, v in REPORT['env'].items() if k != 'torch_config'})
del torch

# 4) memory sampler (every 5 s): GPU via nvidia-smi, CPU = RSS sum of the training process tree and system used
import psutil
PEAK = dict(gpu_mem_used_mib=0, proc_tree_rss_gb=0.0, system_used_gb=0.0, samples=0)
STOP_SAMPLER = threading.Event()
def sampler(pid):
    while not STOP_SAMPLER.is_set():
        try:
            g = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                               capture_output=True, text=True, timeout=10).stdout.split()
            PEAK['gpu_mem_used_mib'] = max([PEAK['gpu_mem_used_mib']] + [int(x) for x in g])
        except Exception:
            pass
        try:
            root = psutil.Process(pid); procs = [root] + root.children(recursive=True); rss = 0
            for p in procs:
                try:
                    rss += p.memory_info().rss
                except psutil.Error:
                    pass
            PEAK['proc_tree_rss_gb'] = max(PEAK['proc_tree_rss_gb'], rss / 2**30)
        except psutil.Error:
            pass
        vm = psutil.virtual_memory()
        PEAK['system_used_gb'] = max(PEAK['system_used_gb'], (vm.total - vm.available) / 2**30)
        PEAK['samples'] += 1
        STOP_SAMPLER.wait(5)

# 5) train (kit script unchanged; only --holdout '' --hours 1000 --max_steps N)
cmd = [sys.executable, '-u', str(TRAIN), '--code', str(CODE), '--run', str(RUN)] + FIXED_ARGS + ['--max_steps', str(STEPS)]
REPORT['command'] = cmd
LOG = OUT / 'train_fpnet.log'
lt = time.localtime(); start_sec = lt.tm_hour * 3600 + lt.tm_min * 60 + lt.tm_sec; t_start = time.time()
with open(LOG, 'w') as lf:
    proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True,
                            env=dict(os.environ, NUMBA_CACHE_DIR=str(OUT / 'numba_cache')))
    th = threading.Thread(target=sampler, args=(proc.pid,), daemon=True); th.start()
    guard_hit = False
    while proc.poll() is None:
        time.sleep(10)
        if SMOKE and time.time() - T0 > GUARD_SECONDS:
            guard_hit = True; log('GUARD: stopping the training process group at', round(time.time() - T0), 's')
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()
            break
    try:
        os.killpg(proc.pid, signal.SIGKILL)   # daemon data workers left behind after a normal exit
    except ProcessLookupError:
        pass
STOP_SAMPLER.set(); th.join(timeout=20)
REPORT['returncode'] = proc.returncode; REPORT['guard_hit'] = guard_hit
REPORT['seconds_train_process'] = round(time.time() - t_start, 1)
REPORT['peak'] = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in PEAK.items()}
text = LOG.read_text()
print(text[-3000:])

# 6) summary + extrapolation (pre-registered formula, see fpnet_full_report)
ns = {}
exec(compile(REPORT_SRC, 'fpnet_full_report', 'exec'), ns)
REPORT['summary'] = ns['summarize'](text, start_sec)
REPORT['seconds_total_notebook'] = round(time.time() - T0, 1)
done = 'ALL_DONE' in text
REPORT['status'] = 'ok' if (done and proc.returncode == 0 and not guard_hit) else ('guard_stopped' if guard_hit else 'failed')
for name in ('fpnet_history.json',):
    if (RUN / name).exists():
        shutil.copy(RUN / name, OUT / name)
if SMOKE:
    for p in RUN.glob('fpnet_*.pt'):
        p.unlink()      # smoke weights are not kept (pre-registered)
else:
    for name in ('fpnet_final.pt', 'fpnet_last.pt'):
        if (RUN / name).exists():
            shutil.copy(RUN / name, OUT / name)
            REPORT[f'{name}_sha256'] = sha256(OUT / name)
shutil.rmtree(CODE, ignore_errors=True); shutil.rmtree(OUT / 'numba_cache', ignore_errors=True)
shutil.rmtree(WORK, ignore_errors=True)   # only symlinks to /kaggle/input; keep them out of the kernel output
(OUT / ('fpnet_smoke_report.json' if SMOKE else 'fpnet_full_report.json')).write_text(json.dumps(REPORT, indent=2, default=str))
log('REPORT', json.dumps({k: v for k, v in REPORT.items() if k not in ('input_sha256', 'env', 'command')}, default=str)[:4000])
assert REPORT['status'] in ('ok', 'guard_stopped'), REPORT['status']
